# Q3 GPU check (Colab)

Setup and a speed measurement only. **Nothing here trains a model for real**: the one cell that runs
gradient steps is a ~1 minute throughput test, off by default (`RUN_TIMING = False`).

Before running:
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload the folder `core/data/processed/q3_panel/` from your laptop to Google Drive at
   `MyDrive/fedci/q3_panel/` (about 700 MB). It is built from licensed CRSP/Compustat data:
   check your school's WRDS terms allow it on Google Drive, and never share the folder.
3. If the GitHub repo is private, add a Colab secret `GITHUB_TOKEN` (key icon in the left bar)
   with a read-only token. Never paste a token into a cell.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "github.com/RileyK05/ml_fed_causal_inference.git"
BRANCH = "q3/cde"  # the branch with the real-panel code
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
if not os.path.exists("/content/repo"):
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/repo"], capture_output=True, text=True)
    print(r.returncode, r.stderr.replace(token, "***") if token else r.stderr)
%cd /content/repo

In [ ]:
# torch is preinstalled on Colab; this adds the project packages and torchcde.
!pip install -q -e "core[q3]" -e "encoder[dl]" -e "cde[dl]"

In [ ]:
import shutil, pathlib
src = pathlib.Path("/content/drive/MyDrive/fedci/q3_panel")
dst = pathlib.Path("/content/repo/core/data/processed/q3_panel")
assert src.exists(), f"upload the panel folder to {src} first"
if not dst.exists():
    shutil.copytree(src, dst)
print(sorted(p.name for p in dst.iterdir()))

In [ ]:
from fedcore.q3 import resolve_device
from fedcore.q3.real import load_real_panel

panel, rows, meta = load_real_panel()  # raw WRDS files are absent here, so the cache loads as is
device = resolve_device()
print("device:", device)
print(f"{meta['n_rows']:,} rows | {meta['n_meetings']} meetings | {meta['n_firms']} firms | built {meta['built']} (v{meta['build_version']})")
print("history", panel.history.shape, "| fundamentals", panel.fundamentals.shape, "| context", panel.context.shape)

## Speed test (trains briefly; off by default)

Times forward + backward steps at the real batch size (4 meetings x ~500 firms) for the
transformer and the log-signature CDE. Nothing is saved; the weights are thrown away. Use the
numbers to size the real runs. Set `RUN_TIMING = True` only after agreeing the training plan.

In [ ]:
RUN_TIMING = False

if RUN_TIMING:
    import time
    import numpy as np
    import torch
    from fedcore.q3 import ResponseModel
    from fedcore.q3.train import ChannelScaler
    from fedenc.encoder_transformer import PatchTransformerEncoder
    from fedcde.encoder_cde import CDELogSigEncoder

    first = panel.meetings()[:4]
    batch = panel.subset(np.isin(panel.meeting, first.to_numpy()))
    scaler = ChannelScaler.fit(batch)
    s = scaler.transform(batch)
    t = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt, device=device)
    x, m = t(s["history"]), t(batch.history_mask, torch.bool)
    f, fm = t(s["fundamentals"]), t(batch.fundamentals_mask, torch.bool)
    c, sh, y = t(s["context"]), t(batch.shock), t(batch.target)

    for name, enc in (("transformer", PatchTransformerEncoder()), ("cde_logsig", CDELogSigEncoder(window=252, window_size=10))):
        model = ResponseModel(enc, n_fund=f.shape[1], n_ctx=c.shape[1]).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        for step in range(13):  # 3 warm-up steps, then 10 timed
            if step == 3:
                torch.cuda.synchronize() if device.type == "cuda" else None
                t0 = time.perf_counter()
            mu, _, _ = model(x, m, f, fm, c, sh)
            loss = ((mu - y) ** 2).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        torch.cuda.synchronize() if device.type == "cuda" else None
        per = (time.perf_counter() - t0) / 10
        n_params = sum(p.numel() for p in model.parameters())
        print(f"{name:12s} {n_params:,} params | {per:.3f} s/step at {len(y)} rows | ~{per * meta['n_meetings'] / 4:.0f} s per pass over all meetings")